This notebook demonstrates how to install local Python packages at build-time, include local Python modules at run-time, and bundle local static data used by the notebook.

In [3]:
factor = 2

xcengine_config = dict(
    workflow_id="inclusions",
    environment_file="../environment.yml",
    container_image_tag="inclusions:1",
    include_directory=True,  # include the whole parent directory of the notebook
    build_includes=["../mylocalpackage"]  # install these local packages in the build
)

In [ ]:
import csv
import pathlib
import xarray as xr

Import some simple demonstration functions from our local Python modules.

In [ ]:
from mymodule import addone
from mylocalpackage import multiply

Reading local data works slightly differently in the notebook and the generated EOAP script. In the notebook, we can usually assume that the current directory (CWD) is the notebook's directory and look for data there. In an EOAP, the current directory is usually not the same as the script's directory, so we have to explicitly define the data directory relative to the script's own path.

In [14]:
try:
    # Find the script's parent dir, if we're running as a script.
    datadir = pathlib.Path(__file__).parent
except NameError:
    # If __file__ is not defined, assume we're running in a notebook.
    datadir = pathlib.Path.cwd()
datadir

PosixPath('/home/pont/loc/repos/xcengine/examples/inclusions/notebook')

When reading local data, we explicitly use `datadir` as the parent path.

In [5]:
with open(datadir / "input-data.csv", newline="") as fh:
    csv_data = [float(row[0]) for row in csv.reader(fh)]

In [6]:
csv_data

[1.0, 3.0, 7.0, 13.0, 23.0, 42.0]

Process the data using the functions we imported from our local code.

In [7]:
modified_data = [addone(multiply(x, factor)) for x in csv_data]

Create a simple xarray Dataset from the data.

In [8]:
ds = xr.Dataset(
    {
        "v": (["t"], csv_data),
    },
    coords={
        "t": modified_data,
    },
)

ds

<xarray.Dataset> Size: 96B
Dimensions:  (t: 6)
Coordinates:
  * t        (t) float64 48B 3.0 7.0 15.0 27.0 47.0 85.0
Data variables:
    v        (t) float64 48B 1.0 3.0 7.0 13.0 23.0 42.0

The dataset `ds` will be automatically found and written by xcengine.